# Physics-Informed Neural Networks (PINNs) for Geodynamic Modeling

This notebook implements PINNs for volcanic and seismic physics modeling, incorporating governing equations and physical constraints into the neural network training process.

## 1. Setup and Imports

In [ ]:
import tensorflow as tf
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tensorflow import keras
from tensorflow.keras import layers
import scipy.sparse as sp
from datetime import datetime, timedelta

# Set random seeds for reproducibility
np.random.seed(42)
tf.random.set_seed(42)

# Detect hardware
print("TensorFlow version:", tf.__version__)
print("Available devices:")
for device in tf.config.list_physical_devices():
    print(f"  {device.device_type}: {device.name}")

## 2. PINN Base Architecture

In [ ]:
class PINN(tf.keras.Model):
    """
    Base Physics-Informed Neural Network class
    """
    def __init__(self, layers_config, activation='tanh'):
        super(PINN, self).__init__()
        
        # Build the neural network
        self.hidden_layers = []
        for units in layers_config:
            self.hidden_layers.append(layers.Dense(units, activation=activation,
                                                  kernel_initializer='glorot_normal'))
        
        # Output layer (no activation for regression)
        self.output_layer = layers.Dense(1, kernel_initializer='glorot_normal')
    
    def call(self, inputs):
        x = inputs
        for layer in self.hidden_layers:
            x = layer(x)
        return self.output_layer(x)


class SeismicWavePINN(PINN):
    """
    PINN for seismic wave propagation
    Solves the wave equation: ∂²u/∂t² = c²∇²u
    """
    def __init__(self, layers_config, wave_speed=3.0):
        super().__init__(layers_config)
        self.c = tf.constant(wave_speed, dtype=tf.float32)  # km/s
    
    @tf.function
    def physics_loss(self, x, y, z, t):
        """
        Compute physics loss based on wave equation
        """
        with tf.GradientTape(persistent=True) as tape:
            tape.watch([x, y, z, t])
            
            # Stack inputs
            inputs = tf.stack([x, y, z, t], axis=1)
            
            # Forward pass
            u = self(inputs)
            
            # First derivatives
            u_t = tape.gradient(u, t)
            u_x = tape.gradient(u, x)
            u_y = tape.gradient(u, y)
            u_z = tape.gradient(u, z)
            
        # Second derivatives
        with tf.GradientTape(persistent=True) as tape2:
            tape2.watch([x, y, z, t])
            u_tt = tape.gradient(u_t, t)
            u_xx = tape.gradient(u_x, x)
            u_yy = tape.gradient(u_y, y)
            u_zz = tape.gradient(u_z, z)
        
        # Wave equation residual
        wave_residual = u_tt - self.c**2 * (u_xx + u_yy + u_zz)
        
        return tf.reduce_mean(tf.square(wave_residual))


class MagmaChamberPINN(PINN):
    """
    PINN for magma chamber pressure evolution
    Based on Mogi model for volcanic deformation
    """
    def __init__(self, layers_config, chamber_depth=5.0, poisson_ratio=0.25):
        super().__init__(layers_config)
        self.depth = tf.constant(chamber_depth, dtype=tf.float32)  # km
        self.nu = tf.constant(poisson_ratio, dtype=tf.float32)
    
    @tf.function
    def physics_loss(self, x, y, t):
        """
        Compute physics loss based on Mogi model
        Surface deformation: u_z = (1-ν) * ΔV * d / (π * r³)
        where r = sqrt(x² + y² + d²)
        """
        with tf.GradientTape(persistent=True) as tape:
            tape.watch([x, y, t])
            
            # Stack inputs
            inputs = tf.stack([x, y, t], axis=1)
            
            # Forward pass (predicts vertical deformation)
            u_z = self(inputs)
            
            # Compute radial distance from source
            r = tf.sqrt(x**2 + y**2 + self.depth**2)
            
            # Mogi model prediction (simplified, ΔV absorbed in network)
            mogi_factor = (1 - self.nu) * self.depth / (np.pi * r**3)
            
            # Time derivative for transient behavior
            u_t = tape.gradient(u_z, t)
        
        # Physics constraint: deformation pattern should follow Mogi model
        # and temporal evolution should be smooth
        spatial_loss = tf.reduce_mean(tf.square(tape.gradient(u_z / mogi_factor, x)**2 + 
                                               tape.gradient(u_z / mogi_factor, y)**2))
        temporal_loss = tf.reduce_mean(tf.square(tape.gradient(u_t, t)))  # Smooth temporal evolution
        
        return spatial_loss + 0.1 * temporal_loss


class ThermalDiffusionPINN(PINN):
    """
    PINN for thermal diffusion in volcanic systems
    Solves: ∂T/∂t = α∇²T + Q(x,y,z,t)
    """
    def __init__(self, layers_config, thermal_diffusivity=1e-6):
        super().__init__(layers_config)
        self.alpha = tf.constant(thermal_diffusivity, dtype=tf.float32)  # m²/s
    
    @tf.function
    def physics_loss(self, x, y, z, t):
        """
        Compute physics loss based on heat equation
        """
        with tf.GradientTape(persistent=True) as tape:
            tape.watch([x, y, z, t])
            
            # Stack inputs
            inputs = tf.stack([x, y, z, t], axis=1)
            
            # Forward pass (temperature prediction)
            T = self(inputs)
            
            # First derivatives
            T_t = tape.gradient(T, t)
            T_x = tape.gradient(T, x)
            T_y = tape.gradient(T, y)
            T_z = tape.gradient(T, z)
            
        # Second derivatives
        with tf.GradientTape(persistent=True) as tape2:
            tape2.watch([x, y, z])
            T_xx = tape.gradient(T_x, x)
            T_yy = tape.gradient(T_y, y)
            T_zz = tape.gradient(T_z, z)
        
        # Heat equation residual (without source term for now)
        heat_residual = T_t - self.alpha * (T_xx + T_yy + T_zz)
        
        return tf.reduce_mean(tf.square(heat_residual))

## 3. Training Functions

In [ ]:
def create_training_data_seismic(n_samples=10000, domain_size=10.0):
    """
    Create synthetic training data for seismic wave PINN
    """
    # Random points in space-time domain
    x = np.random.uniform(-domain_size, domain_size, n_samples)
    y = np.random.uniform(-domain_size, domain_size, n_samples)
    z = np.random.uniform(0, domain_size, n_samples)  # Depth positive
    t = np.random.uniform(0, 10, n_samples)  # 10 seconds
    
    # Convert to tensors
    x_tf = tf.constant(x, dtype=tf.float32)
    y_tf = tf.constant(y, dtype=tf.float32)
    z_tf = tf.constant(z, dtype=tf.float32)
    t_tf = tf.constant(t, dtype=tf.float32)
    
    # Initial condition: Gaussian pulse at origin
    u_data = np.exp(-0.5 * (x**2 + y**2 + z**2)) * np.cos(2*np.pi*t)
    u_tf = tf.constant(u_data, dtype=tf.float32)
    
    return x_tf, y_tf, z_tf, t_tf, u_tf


def create_training_data_magma(n_samples=5000, domain_size=20.0):
    """
    Create synthetic training data for magma chamber PINN
    """
    # Surface points around volcano
    x = np.random.uniform(-domain_size, domain_size, n_samples)
    y = np.random.uniform(-domain_size, domain_size, n_samples)
    t = np.random.uniform(0, 365, n_samples)  # One year in days
    
    # Convert to tensors
    x_tf = tf.constant(x, dtype=tf.float32)
    y_tf = tf.constant(y, dtype=tf.float32)
    t_tf = tf.constant(t, dtype=tf.float32)
    
    # Synthetic deformation data (Mogi-like pattern)
    r = np.sqrt(x**2 + y**2 + 5.0**2)  # 5km depth
    u_data = 0.1 * (1 + 0.5*np.sin(2*np.pi*t/365)) / r  # Seasonal variation
    u_tf = tf.constant(u_data, dtype=tf.float32)
    
    return x_tf, y_tf, t_tf, u_tf


@tf.function
def train_step_pinn(model, optimizer, x, y, z, t, u_data=None, data_weight=1.0, physics_weight=1.0):
    """
    Single training step for PINN
    """
    with tf.GradientTape() as tape:
        # Data loss (if available)
        data_loss = 0.0
        if u_data is not None:
            if z is not None:
                inputs = tf.stack([x, y, z, t], axis=1)
            else:
                inputs = tf.stack([x, y, t], axis=1)
            u_pred = model(inputs)
            data_loss = tf.reduce_mean(tf.square(u_pred - tf.reshape(u_data, [-1, 1])))
        
        # Physics loss
        if z is not None:
            physics_loss = model.physics_loss(x, y, z, t)
        else:
            physics_loss = model.physics_loss(x, y, t)
        
        # Total loss
        total_loss = data_weight * data_loss + physics_weight * physics_loss
    
    # Compute gradients and update
    gradients = tape.gradient(total_loss, model.trainable_variables)
    optimizer.apply_gradients(zip(gradients, model.trainable_variables))
    
    return total_loss, data_loss, physics_loss


def train_pinn(model, x, y, z, t, u_data=None, epochs=1000, learning_rate=0.001):
    """
    Train PINN model
    """
    optimizer = tf.keras.optimizers.Adam(learning_rate=learning_rate)
    
    history = {'loss': [], 'data_loss': [], 'physics_loss': []}
    
    for epoch in range(epochs):
        loss, data_loss, physics_loss = train_step_pinn(
            model, optimizer, x, y, z, t, u_data
        )
        
        history['loss'].append(float(loss))
        history['data_loss'].append(float(data_loss))
        history['physics_loss'].append(float(physics_loss))
        
        if epoch % 100 == 0:
            print(f"Epoch {epoch}: Loss = {loss:.6f}, Data = {data_loss:.6f}, Physics = {physics_loss:.6f}")
    
    return history

## 4. Train Seismic Wave PINN

In [ ]:
# Create seismic wave PINN
seismic_pinn = SeismicWavePINN([64, 64, 64, 64], wave_speed=3.0)

# Generate training data
x_seis, y_seis, z_seis, t_seis, u_seis = create_training_data_seismic(n_samples=5000)

# Train the model
print("Training Seismic Wave PINN...")
seismic_history = train_pinn(
    seismic_pinn, x_seis, y_seis, z_seis, t_seis, u_seis,
    epochs=2000, learning_rate=0.001
)

# Plot training history
plt.figure(figsize=(12, 4))
plt.subplot(1, 3, 1)
plt.plot(seismic_history['loss'])
plt.title('Total Loss')
plt.xlabel('Epoch')
plt.yscale('log')

plt.subplot(1, 3, 2)
plt.plot(seismic_history['data_loss'])
plt.title('Data Loss')
plt.xlabel('Epoch')
plt.yscale('log')

plt.subplot(1, 3, 3)
plt.plot(seismic_history['physics_loss'])
plt.title('Physics Loss')
plt.xlabel('Epoch')
plt.yscale('log')

plt.tight_layout()
plt.show()

## 5. Train Magma Chamber PINN

In [ ]:
# Create magma chamber PINN
magma_pinn = MagmaChamberPINN([32, 32, 32], chamber_depth=5.0)

# Generate training data
x_magma, y_magma, t_magma, u_magma = create_training_data_magma(n_samples=3000)

# Train the model
print("Training Magma Chamber PINN...")
magma_history = train_pinn(
    magma_pinn, x_magma, y_magma, None, t_magma, u_magma,
    epochs=1500, learning_rate=0.001
)

# Visualize deformation pattern
grid_size = 50
x_grid = np.linspace(-20, 20, grid_size)
y_grid = np.linspace(-20, 20, grid_size)
X, Y = np.meshgrid(x_grid, y_grid)
T = np.ones_like(X) * 180  # Mid-year

# Predict deformation
inputs_grid = tf.stack([
    tf.constant(X.flatten(), dtype=tf.float32),
    tf.constant(Y.flatten(), dtype=tf.float32),
    tf.constant(T.flatten(), dtype=tf.float32)
], axis=1)

U_pred = magma_pinn(inputs_grid).numpy().reshape(grid_size, grid_size)

plt.figure(figsize=(10, 8))
plt.contourf(X, Y, U_pred, levels=20, cmap='RdBu_r')
plt.colorbar(label='Vertical deformation (m)')
plt.xlabel('X (km)')
plt.ylabel('Y (km)')
plt.title('Predicted Surface Deformation Pattern')
plt.axis('equal')
plt.show()

## 6. Coupled PINN for Multi-Physics

In [ ]:
class CoupledVolcanicPINN(tf.keras.Model):
    """
    Coupled PINN for volcanic system modeling
    Combines thermal, mechanical, and seismic physics
    """
    def __init__(self, layers_config):
        super().__init__()
        
        # Separate networks for different physics
        self.thermal_net = PINN(layers_config)
        self.deformation_net = PINN(layers_config)
        self.seismic_net = PINN(layers_config)
        
        # Coupling network
        self.coupling_net = PINN([32, 32])
        
        # Physical parameters
        self.alpha = tf.constant(1e-6, dtype=tf.float32)  # Thermal diffusivity
        self.E = tf.constant(50e9, dtype=tf.float32)  # Young's modulus (Pa)
        self.nu = tf.constant(0.25, dtype=tf.float32)  # Poisson's ratio
    
    @tf.function
    def call(self, x, y, z, t):
        """
        Forward pass through coupled system
        """
        inputs = tf.stack([x, y, z, t], axis=1)
        
        # Individual physics predictions
        T = self.thermal_net(inputs)  # Temperature
        u = self.deformation_net(inputs)  # Deformation
        v = self.seismic_net(inputs)  # Seismic velocity
        
        # Coupling effects
        coupling_input = tf.concat([T, u, v], axis=1)
        coupling = self.coupling_net(coupling_input)
        
        return T, u, v, coupling
    
    @tf.function
    def physics_loss(self, x, y, z, t):
        """
        Combined physics loss with coupling
        """
        with tf.GradientTape(persistent=True) as tape:
            tape.watch([x, y, z, t])
            
            # Forward pass
            T, u, v, coupling = self(x, y, z, t)
            
            # Temperature gradients
            T_t = tape.gradient(T, t)
            T_x = tape.gradient(T, x)
            T_y = tape.gradient(T, y)
            T_z = tape.gradient(T, z)
            
            # Deformation gradients
            u_x = tape.gradient(u, x)
            u_y = tape.gradient(u, y)
            u_z = tape.gradient(u, z)
            
            # Velocity gradients
            v_t = tape.gradient(v, t)
            v_x = tape.gradient(v, x)
            
        # Second derivatives
        with tf.GradientTape() as tape2:
            tape2.watch([x, y, z])
            T_xx = tape.gradient(T_x, x)
            T_yy = tape.gradient(T_y, y)
            T_zz = tape.gradient(T_z, z)
            v_xx = tape.gradient(v_x, x)
        
        # Physics residuals
        # 1. Heat equation with source from deformation
        heat_source = self.E * coupling * (u_x + u_y + u_z)  # Volumetric strain heating
        heat_residual = T_t - self.alpha * (T_xx + T_yy + T_zz) - heat_source
        
        # 2. Elastic deformation with thermal expansion
        thermal_strain = 1e-5 * T  # Thermal expansion coefficient
        deform_residual = u_x + u_y + u_z - thermal_strain
        
        # 3. Seismic velocity changes with temperature
        velocity_temp_coupling = -0.001 * T  # Velocity decreases with temperature
        seismic_residual = v_t - 3.0**2 * v_xx - velocity_temp_coupling
        
        # Combined loss
        total_physics_loss = (
            tf.reduce_mean(tf.square(heat_residual)) +
            tf.reduce_mean(tf.square(deform_residual)) +
            tf.reduce_mean(tf.square(seismic_residual))
        )
        
        return total_physics_loss


# Create and demonstrate coupled PINN
print("Creating Coupled Volcanic PINN...")
coupled_pinn = CoupledVolcanicPINN([64, 64, 64])

# Generate test data
n_test = 1000
x_test = tf.random.uniform([n_test], -10, 10)
y_test = tf.random.uniform([n_test], -10, 10)
z_test = tf.random.uniform([n_test], 0, 10)
t_test = tf.random.uniform([n_test], 0, 365)

# Forward pass
T_pred, u_pred, v_pred, coupling_pred = coupled_pinn(x_test, y_test, z_test, t_test)

print(f"Temperature predictions shape: {T_pred.shape}")
print(f"Deformation predictions shape: {u_pred.shape}")
print(f"Velocity predictions shape: {v_pred.shape}")
print(f"Coupling predictions shape: {coupling_pred.shape}")

## 7. Integration with Data-Driven Models

In [ ]:
class HybridPINNPredictor:
    """
    Combines PINN physics models with data-driven ML models
    """
    def __init__(self, pinn_models, ml_models):
        self.pinn_models = pinn_models  # Dict of PINN models
        self.ml_models = ml_models  # Dict of ML models from main training
    
    def predict_volcanic_hazard(self, location, time, observations):
        """
        Comprehensive hazard prediction combining physics and data
        """
        x, y, z = location
        
        # PINN predictions (physics-based)
        physics_predictions = {}
        
        if 'seismic' in self.pinn_models:
            wave_field = self.pinn_models['seismic'](
                tf.constant([[x, y, z, time]], dtype=tf.float32)
            )
            physics_predictions['wave_amplitude'] = float(wave_field[0, 0])
        
        if 'magma' in self.pinn_models:
            deformation = self.pinn_models['magma'](
                tf.constant([[x, y, time]], dtype=tf.float32)
            )
            physics_predictions['surface_deformation'] = float(deformation[0, 0])
        
        # ML predictions (data-driven)
        ml_predictions = {}
        
        if 'seismic_lstm' in self.ml_models and 'seismic_data' in observations:
            seismic_prob = self.ml_models['seismic_lstm'].predict(
                observations['seismic_data']
            )
            ml_predictions['seismic_event_probability'] = float(seismic_prob[0])
        
        if 'thermal_cnn' in self.ml_models and 'thermal_image' in observations:
            thermal_class = self.ml_models['thermal_cnn'].predict(
                observations['thermal_image']
            )
            ml_predictions['thermal_anomaly_class'] = int(np.argmax(thermal_class[0]))
        
        # Combine predictions
        hazard_score = self._compute_hazard_score(physics_predictions, ml_predictions)
        
        return {
            'physics_predictions': physics_predictions,
            'ml_predictions': ml_predictions,
            'hazard_score': hazard_score,
            'timestamp': datetime.now().isoformat()
        }
    
    def _compute_hazard_score(self, physics, ml):
        """
        Combine physics and ML predictions into unified hazard score
        """
        score = 0.0
        
        # Physics-based contributions
        if 'wave_amplitude' in physics:
            score += 0.2 * min(abs(physics['wave_amplitude']) / 0.1, 1.0)
        
        if 'surface_deformation' in physics:
            score += 0.2 * min(abs(physics['surface_deformation']) / 0.01, 1.0)
        
        # ML-based contributions
        if 'seismic_event_probability' in ml:
            score += 0.3 * ml['seismic_event_probability']
        
        if 'thermal_anomaly_class' in ml:
            score += 0.3 * (ml['thermal_anomaly_class'] / 4.0)  # Normalize by max class
        
        return min(score, 1.0)


# Example usage
print("\nDemonstrating Hybrid PINN-ML Prediction:")

# Mock ML models (in practice, load from saved files)
mock_ml_models = {
    'seismic_lstm': type('MockModel', (), {'predict': lambda self, x: np.array([[0.75]])}),
    'thermal_cnn': type('MockModel', (), {'predict': lambda self, x: np.array([[0.1, 0.2, 0.5, 0.2]])})
}

# Create hybrid predictor
hybrid = HybridPINNPredictor(
    pinn_models={'seismic': seismic_pinn, 'magma': magma_pinn},
    ml_models=mock_ml_models
)

# Make prediction
result = hybrid.predict_volcanic_hazard(
    location=(5.0, 5.0, 2.0),
    time=180.0,
    observations={
        'seismic_data': np.random.randn(1, 100, 8),
        'thermal_image': np.random.rand(1, 224, 224, 3)
    }
)

print(json.dumps(result, indent=2))

## 8. Save PINN Models

In [ ]:
# Save PINN models
print("Saving PINN models...")

# Save weights
seismic_pinn.save_weights('seismic_pinn_weights.h5')
magma_pinn.save_weights('magma_pinn_weights.h5')

# Save full models using SavedModel format
tf.saved_model.save(seismic_pinn, 'seismic_pinn_saved_model')
tf.saved_model.save(magma_pinn, 'magma_pinn_saved_model')

# Save training configuration
pinn_config = {
    'seismic_pinn': {
        'layers': [64, 64, 64, 64],
        'wave_speed': 3.0,
        'domain_size': 10.0
    },
    'magma_pinn': {
        'layers': [32, 32, 32],
        'chamber_depth': 5.0,
        'poisson_ratio': 0.25
    },
    'training': {
        'epochs': 2000,
        'learning_rate': 0.001,
        'physics_weight': 1.0,
        'data_weight': 1.0
    }
}

import json
with open('pinn_config.json', 'w') as f:
    json.dump(pinn_config, f, indent=2)

print("\nPINN models saved successfully!")
print("Files created:")
print("  - seismic_pinn_weights.h5")
print("  - magma_pinn_weights.h5")
print("  - seismic_pinn_saved_model/")
print("  - magma_pinn_saved_model/")
print("  - pinn_config.json")